# Assignment 1: Data Profiling (EDA) Report
**Dataset:** Adult Census Income (UCI)

## Team Members
| Name | Matric No. | Role |
|---|---|---|
| Mohamad Azamuddin | A209604 | Project Lead: Sections 1, 2 and setup |
| Muhammad Qinawee | A210093 | Risk Analysis & Integration: Sections 5, 6, final run-through and submission |
| Nik Aiman Ashraf| <MATRIC> | Data Profiling & Quality Remediation: Section 3 |
| Airell Mikhail | A210140 | Data Visualization & Diagnostic Charts: Section 4 |

## How to Run
1. Clone the repository and open a terminal in its folder.
2. `pip install -r requirements.txt`
3. `jupyter notebook adult_data_profiling.ipynb`
4. Run **Kernel > Restart & Run All**. The data is downloaded automatically (internet required). The cleaned output is saved to `data/adult_clean.csv`.

## 1. Scenario
A government agency or NGO wants to understand which demographic and employment factors are associated with earning more than $50K per year. The Adult Census Income dataset fits this assignment because it contains missing values (in `workclass`, `occupation` and `native-country`), outliers and suspicious values (e.g. `capital-gain`, `hours-per-week`), sensitive attributes (`sex`, `race`, `native-country`) and an imbalanced target.

**Diagnostic questions:**
1. Where are the missing values, and are they random?
2. Which numeric fields contain outliers or invalid values?
3. Does the data carry PII or bias risks that would affect fairness if used for modelling?

## 2. Data Source
- **Dataset:** Adult Census Income, UCI Machine Learning Repository
- **URL:** https://archive.ics.uci.edu/dataset/2/adult
- **Origin:** Extracted from the 1994 US Census database (Becker and Kohavi, 1996)
- **Size:** about 48,842 rows, 14 features and 1 target (`income`)
- **How obtained:** downloaded programmatically with the `ucimlrepo` package (`fetch_ucirepo(id=2)`). No scraping.
- **License:** CC BY 4.0 (verify on the UCI page)
- **Variables:** raw data = `df_raw`, cleaned data = `df_clean`, saved to `data/adult_clean.csv`

In [1]:
import pandas as pd
from ucimlrepo import fetch_ucirepo

adult = fetch_ucirepo(id=2)
df_raw = pd.concat([adult.data.features, adult.data.targets], axis=1)
print(df_raw.shape)
df_raw.head()

(48842, 15)


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


## 3. Data Profiling and Issue Handling
*(teammate)*

### 3.1 Missing values
### 3.2 Outliers
### 3.3 Other quality issues

In [2]:
# df_clean = ...
# df_clean.to_csv('data/adult_clean.csv', index=False)

## 4. Exploratory (Diagnostic) Charts
*(teammate: at least 2 labeled charts)*

### 5.1 PII (Personally Identifiable Information) Risk

**Risk:** The dataset contains no direct identifiers (no names, ID numbers, addresses or phone numbers). However, it contains quasi-identifiers: `age`, `sex`, `race`, `native-country`, `marital-status`, `education` and `occupation`. Combined, they can single out individuals, especially people in rare groups (e.g. an older woman from a small country with an uncommon occupation).

**Evidence:** Sensitive attributes (race, sex, marital status, country of origin) are present for every record, and `native-country` has many rare categories with very few rows.

**Impact:** Linking this data with another dataset could re-identify individuals and reveal their income bracket.

**Mitigation:** (1) Generalise `age` into bands (e.g. 17-25, 26-35). (2) Group `native-country` into regions and merge rare countries into "Other". (3) Drop `fnlwgt` and any column not needed for the scenario. (4) Do not publish row-level data beyond this public dataset. (5) Check that every combination of quasi-identifiers appears in at least k records (k-anonymity).

### 5.2 Sampling Bias

**Risk:** The data was extracted from the 1994 US Census (Becker and Kohavi, 1996) using filters on age, income and working hours, so it is neither current nor representative of other countries or of the full population.

**Evidence:** In the raw data, 89.7% of records are from the United States and 85.5% are White. Black individuals make up 9.6%, Asian-Pacific Islanders 3.1%, American Indian/Eskimo 1.0% and Other 0.8%. Non-US countries and minority groups therefore appear in very small numbers.

**Impact:** Findings may not generalise to other countries, ethnic groups or the present day. Wages and job structures from 1994 are outdated.

**Mitigation:** State clearly that conclusions apply only to the 1994 US context, avoid generalising to other populations, and collect newer, more diverse data before any real decision-making.

### 5.3 Imbalanced Groups

**Risk:** Both the target and the sensitive groups are imbalanced.

**Evidence:** Only 23.9% of records earn >50K versus 76.1% earning <=50K. Males make up 66.8% of records and females 33.2%. About 30.4% of men earn >50K compared to 10.9% of women, so `sex` is strongly linked to the target.

**Impact:** A model or analysis can favour the majority group, report misleadingly high overall accuracy, and learn historical gender or racial inequality as if it were a rule.

**Mitigation:** Use stratified sampling, resampling or class weights; report metrics per group (precision and recall by sex and race) instead of overall accuracy only; and avoid using `sex` and `race` directly as predictors unless the scenario explicitly requires it.

### 5.4 Missing Data as a Hidden Bias

**Risk:** Missing values in `workclass`, `occupation` and `native-country` may not be random.

**Impact:** Dropping these rows could remove certain groups (e.g. unemployed or less-documented workers) and shift the group proportions.

**Mitigation:** Compare group proportions before and after cleaning (see Section 3) and keep "missing" as its own category where appropriate.

*Note: percentages above are from the raw data (`df_raw`) before cleaning.*

In [14]:
high = df_raw['income'].str.contains('>50K')

print(df_raw['sex'].value_counts(normalize=True).round(3))
print(df_raw['race'].value_counts(normalize=True).round(3))
print('US:', (df_raw['native-country'] == 'United-States').mean().round(3))
print('>50K:', high.mean().round(3))
print((high.groupby(df_raw['sex']).mean() * 100).round(1))

sex
Male      0.668
Female    0.332
Name: proportion, dtype: float64
race
White                 0.855
Black                 0.096
Asian-Pac-Islander    0.031
Amer-Indian-Eskimo    0.010
Other                 0.008
Name: proportion, dtype: float64
US: 0.897
>50K: 0.239
sex
Female    10.9
Male      30.4
Name: income, dtype: float64


## 6. Conclusion
**Fitness for the scenario:** The Adult Census dataset is **partially fit** for the scenario. It is large, well documented and contains relevant demographic and employment variables, so it is suitable for exploring which factors are associated with earning more than $50K. However, it is **not fit for real-world decisions** about current populations, because the data is from 1994 and demographically narrow.

**Answers to the diagnostic questions:**
1. *Missing values:* located in `workclass`, `occupation` and `native-country` (see Section 3 for whether they are random).
2. *Outliers and invalid values:* found in numeric fields such as `capital-gain` and `hours-per-week` (see Sections 3 and 4).
3. *PII and bias risks:* the data has no direct identifiers, but quasi-identifiers, sampling bias (1994 US only) and group imbalance (sex, race, income) create fairness risks (see Section 5).

**What was improved:** Missing values were handled and justified, outliers were examined, duplicates and inconsistent formats were fixed, and data types were corrected (see Section 3). The diagnostic charts (Section 4) confirmed the main data quality problems.

**Remaining limitations:**
1. The data is from 1994 and reflects the US only, so it is not representative of today or of other countries.
2. Group imbalances (sex, race, native country and income class) remain and can bias any model or conclusion.
3. Quasi-identifiers could allow re-identification if combined with other data.
4. Imputed or flagged values add uncertainty.
5. Associations found here are not causal.

**Recommendation:** Use this dataset for learning and exploratory analysis only. Report results per group, document all cleaning decisions, and use newer and more representative data before applying findings in practice.